# More explicit non algebraicity proof

We starts with the rank-4 matroid on `{1,...,10}` specified by its 20 nonbases and uses SageMath's linear-subclass enumerator to perform these three application of Ingleton–Main lemma:

1. `{1,3}, {2,4}, {6,7}` and add element `11` — 1 valid extension.
2. `{4,6}, {5,8}, {9,10}` and add element `12` — 1 valid extension.
3. `{5,6}, {4,8}, {7,9}` and add element `13` — 3 valid extensions.

The first two application force a single matroid `M12`; the last application gives the three matroid `C_1, C_2, C_3.`

In [ ]:
from itertools import combinations
from sage.all import *
from sage.matroids.advanced import BasisMatroid
def im_extensions(M, generating_pairs, new_element):
    """
    Enumerate the proper Ingleton–Main extensions for three lines of M.
    """
    pairs = [frozenset(P) for P in generating_pairs]
    lines = [M.closure(P) for P in pairs]

    # Verify hypotheses of the Ingleton–Main lemma.
    assert all(M.rank(L) == 2 for L in lines), "Each set must span a line."
    assert all(M.rank(lines[i] | lines[j]) == 3
               for i, j in combinations(range(3), 2)), \
           "Every two lines must be coplanar."
    assert M.rank(frozenset().union(*lines)) == 4, \
           "The three lines must not all be coplanar."
    assert not set.intersection(*(set(L) for L in lines)), \
           "The three lines must not already have a common point."

    # Enumerates the linear subclasses compatible with the
    # condition that the new element be spanned by all three lines.
    subclasses = list(M.linear_subclasses(subsets=lines))
    candidates = [M._extension(new_element, C) for C in subclasses]
    valid = [N for N in candidates if len(N.loops()) == 0]#remove the loop extension

    return valid

## 1. The original matroid

In [ ]:
NONBASES = [
    {1, 2, 3, 4},
    {1, 2, 5, 6},
    {1, 3, 6, 7},
    {2, 3, 5, 7},
    {2, 4, 6, 7},
    {2, 5, 8, 9},
    {2, 5, 8, 10},
    {2, 5, 9, 10},
    {2, 8, 9, 10},
    {3, 4, 5, 6},
    {3, 4, 5, 8},
    {3, 4, 6, 8},
    {3, 5, 6, 8},
    {4, 5, 6, 8},
    {4, 5, 7, 10},
    {4, 6, 9, 10},
    {4, 7, 8, 9},
    {5, 6, 7, 9},
    {5, 8, 9, 10},
    {6, 7, 8, 10},
]

M10 = BasisMatroid(
    groundset=frozenset(range(1, 11)),
    nonbases=[frozenset(X) for X in NONBASES],
)

assert M10.is_valid()
assert M10.is_simple()
assert M10.full_rank() == 4
assert M10.bases_count() == 190       # binomial(10,4) - 20

print(M10)
print("rank =", M10.full_rank())
print("bases =", M10.bases_count())
print("nonbases =", len(M10.nonbases()))

## 2. First Ingleton–Main application: add 11 on `{1,3}`, `{2,4}`, `{6,7}`

There is exactly one proper extension `M11`.

In [ ]:
step1 = im_extensions(
    M10,
    [{1, 3}, {2, 4}, {6, 7}],
    new_element=11,
)

assert len(step1) == 1
M11 = step1[0]
print("M11 bases:", M11.bases_count())
print("M11 nonbases:", len(M11.nonbases()))

## 3. Second Ingleton–Main application: add 12 on `{4,6}`, `{5,8}`, `{9,10}`

Again there is exactly one proper extension `M12`.

In [ ]:
step2 = im_extensions(
    M11,
    [{4, 6}, {5, 8}, {9, 10}],
    new_element=12,
)

assert len(step2) == 1
M12 = step2[0]
print("M12 bases:", M12.bases_count())
print("M12 nonbases:", len(M12.nonbases()))

## 4. Third Ingleton–Main application: add 13 on `{5,6}`, `{4,8}`, and `{7,9}`

 There are 3 valid extensions.

In [ ]:
step3 = im_extensions(
    M12,
    [{5, 6}, {4, 8}, {7, 9}],
    new_element=13,
)

M13_children = sorted(
    step3,
    key=lambda N: -N.bases_count(),
)

assert len(M13_children) == 3

for i, N in enumerate(M13_children, start=1):
    print(
        f"C_{i}: "
        f"{N.bases_count()} bases, {len(N.nonbases())} nonbases"
    )

## 5. Checking Ingleton–Main at depth 4 for `C_1, C_2,` and `C_3`

In [ ]:
import time
from itertools import chain
from sage.all import *


def DoubleCircuits(M,IngletonMain=False):
    #input: a matroid M
    #       a boolean indicating whether Ingleton-Main is checked or not (then Dress-Lovasz is checked)
    #output: for each double circuit CC:
    #          the double circuit degree d, 
    #          the number of points to be added to the intersection
    #          the circuit closures of the double circuit
    #          the intersection of these circuits
    Md = M.dual()
    for F in Md.flats(Md.full_rank() - 2):
        Mdm = Md.contract(F)
        d=Mdm.simplify().size()
        if IngletonMain:
            goodDegree = (d == 3)
        else:
            goodDegree = (d >= 3)
        if goodDegree:
            circs = [M.closure(C) for C in Mdm.cocircuits()]
            inter = M.groundset()
            for C in circs:
                inter = inter.intersection(C)
            ir = M.rank(inter)
            if d-2 - ir <= 0:
                continue
            yield d, circs, inter
 
 
def DressLovaszExtensions(M, CC):
    #input: a matroid M
    #       a list of circuit closures CC
    #output: the subsets that i must lie in the closure of so that i lies in all circuit closures of CC, and the new element i
    E = M.groundset()
    i = -1
    while i in E:
        i -= 1
    
    return [list(c) for c in CC if len(c) < M.size()], i
 
def RecursiveDressLovaszCondition(M,depth=1,IngletonMain=False):
    #input: a matroid M
    #       an integer depth
    #       a boolean indicating whether Ingleton-Main is checked or not (then Dress-Lovasz is checked)
    #output: the Dress-Lovasz (or Ingleton-Main) condition at depth 'depth' for M
    
    if depth==0:
        return True
    
    for dCC in DoubleCircuits(M,IngletonMain):
        d, CC, inter = dCC
        subsets, i = DressLovaszExtensions(M,CC)
        principal = M.extension(element=i,subsets=subsets)
        if depth == 1:
            DLE = (principal,)
        else:
            DLE = chain((principal,), M.extensions(element=i,subsets=subsets))
        goodCC = False
        for N in DLE:
            if N.rank(inter.union([i])) == N.rank(inter):
                continue
            if RecursiveDressLovaszCondition(N,depth-1,IngletonMain):
                goodCC = True
                break
        if not goodCC:
            return False
    return True
 

C1 = M13_children[0]

t = time.time()
print(RecursiveDressLovaszCondition(C1, depth=4, IngletonMain=True))
print(time.time() - t)

C2 = M13_children[1]

t = time.time()
print(RecursiveDressLovaszCondition(C2, depth=4, IngletonMain=True))
print(time.time() - t)

C3 = M13_children[2]
t = time.time()
print(RecursiveDressLovaszCondition(C3, depth = 4, IngletonMain =True))
print(time.time() - t)